In [ ]:
#pip install ipykernel

In [ ]:
# 1. 仮想環境の中に ipykernel をインストールする（今回はこれが重要）

#!pip install ipykernel



# 2. 仮想環境を Jupyter の選択肢（カーネル）として登録する

# --name は識別用、--display-name はJupyter上で表示される名前です

#!python -m ipykernel install --user --name venv_project3 --display-name "Python (project3_venv)"

In [2]:
#社会工学類kdbをデータベース化

import pandas as pd
import sqlite3
import os

os.makedirs('instance', exist_ok=True)

filename = 'kdb_20251220100333_社会工学類.csv'
df_raw = pd.read_csv(filename, encoding='UTF-8', header=None)
header_row_index = df_raw[df_raw.eq("科目番号").any(axis=1)].index[0]
df = pd.read_csv(filename, encoding='UTF-8', skiprows=header_row_index)

conn = sqlite3.connect('instance/project.db')
df.to_sql('courses', conn, if_exists='replace', index=False)
conn.close()

print(f"成功：ヘッダーを {header_row_index + 1} 行目として認識し、データベースを更新しました。")

成功：ヘッダーを 5 行目として認識し、データベースを更新しました。


In [12]:
#必要な情報のみ抽出（科目名、授業概要、備考）

conn = sqlite3.connect('instance/project.db')

query = "SELECT 科目番号, 科目名, 授業概要, 実施学期, 曜時限, 単位数, 備考 FROM courses"
df_selected = pd.read_sql(query, conn)

df_selected = df_selected.fillna('')

df_selected['combined_text'] = (
    df_selected['科目名'] + " " + 
    df_selected['授業概要'] + " " + 
    df_selected['備考']
)

df_selected.to_sql('processed_courses', conn, if_exists='replace', index=False)
conn.close()

print("データの抽出と統合が正常に完了しました！")

データの抽出と統合が正常に完了しました！


In [4]:
#仮想環境指定コード

import sys
import os

venv_path = r"C:\Users\fmiha\project3\venv\Lib\site-packages"
if venv_path not in sys.path:
    sys.path.insert(0, venv_path)  

try:
    from sudachipy import dictionary
    from sudachipy import tokenizer
    from keybert import KeyBERT
    print("✅ 仮想環境のライブラリを認識しました。解析を開始します。")
except ImportError as e:
    print(f"❌ まだ読み込めません。エラー: {e}")
    print("パスが正しいか確認してください:", venv_path)

✅ 仮想環境のライブラリを認識しました。解析を開始します。


In [29]:
!pip install ipython-sql
!pip install prettytable==3.11.0


   ---------------------------------------- 0/4 [ipython-genutils]
   ---------------------------------------- 0/4 [ipython-genutils]
   ---------- ----------------------------- 1/4 [sqlparse]
   ---------- ----------------------------- 1/4 [sqlparse]
   ---------- ----------------------------- 1/4 [sqlparse]
   ---------- ----------------------------- 1/4 [sqlparse]
   -------------------- ------------------- 2/4 [prettytable]
   ------------------------------ --------- 3/4 [ipython-sql]
   ------------------------------ --------- 3/4 [ipython-sql]
   ---------------------------------------- 4/4 [ipython-sql]

  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.17.0
    Uninstalling prettytable-3.17.0:
      Successfully uninstalled prettytable-3.17.0


In [30]:
%load_ext sql
%sql sqlite:///myex.db
%config SqlMagic.autopandas=False
%config SqlMagic.displaycon=False

def tables():
    tables = %sql select tbl_name from sqlite_master where type='table' or type='view'
    return list(tables)
    
tables()

Done.


[]

In [5]:
# 1. SudachiPyの準備（分割モードC：複合名詞を維持）
tokenizer_obj = dictionary.Dictionary(dict="full").create()
mode = tokenizer.Tokenizer.SplitMode.C

def extract_nouns(text):
    """SudachiPyを用いて名詞・固有名詞のみを抽出する関数"""
    tokens = tokenizer_obj.tokenize(text, mode)
    result = []
    for token in tokens:
        pos = token.part_of_speech()
        if pos[0] == "名詞":
            if pos[1] not in ["数詞", "非自立", "接尾", "代名詞"]:
                result.append(token.normalized_form())
    return " ".join(result)


conn = sqlite3.connect('instance/project.db')
df = pd.read_sql("SELECT * FROM processed_courses", conn)

# 3. 形態素解析の実行（名詞のみにフィルタリング）
print("形態素解析を実行中...")
df['filtered_text'] = df['combined_text'].apply(extract_nouns)

# 4. KeyBERTによるキーワード抽出
kw_model = KeyBERT('paraphrase-multilingual-MiniLM-L12-v2')

def get_keywords(text):
    """授業ごとに上位5つのキーワードを抽出"""
    if not text.strip():
        return ""
    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 1), stop_words=None, top_n=5)
    return ",".join([kw[0] for kw in keywords])

print("キーワード抽出を実行中（これには数分かかる場合があります）...")
df['keywords'] = df['filtered_text'].apply(get_keywords)

# 5. 結果をデータベースに保存
df.to_sql('final_keywords', conn, if_exists='replace', index=False)
conn.close()

print("キーワードの作成が完了し、'final_keywords' テーブルに保存されました。")

形態素解析を実行中...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 6db03a68-79b9-4b3b-bb8c-8e1142aab436)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./README.md
Retrying in 1s [Retry 1/5].


キーワード抽出を実行中（これには数分かかる場合があります）...
キーワードの作成が完了し、'final_keywords' テーブルに保存されました。


In [36]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('instance/project.db')

# 1. 解析結果を読み込む
df_topics = pd.read_sql("SELECT * FROM topic_distributions", conn)

# 2. coursesテーブルの全列を読み込む（列名指定によるエラーを回避）
df_raw = pd.read_sql("SELECT * FROM courses", conn)

conn.close()

# 3. 科目番号で統合
# どちらのテーブルにも「科目名」がある場合、重複を防ぐため片方を削除します
if '科目名' in df_raw.columns and '科目名' in df_topics.columns:
    df_raw = df_raw.drop(columns=['科目名'])

df_final = pd.merge(df_topics, df_raw, on='科目番号', how='left')

# 4. 決定版テーブルとして保存
conn = sqlite3.connect('instance/project.db')
df_final.to_sql('final_recommendation_data', conn, if_exists='replace', index=False)
conn.close()

print("✅ 全情報を統合した 'final_recommendation_data' を作成しました。")

✅ 全情報を統合した 'final_recommendation_data' を作成しました。


In [37]:
import sqlite3
import pandas as pd
import ast

def export_keywords_to_excel(db_path, output_path="授業キーワード確認リスト.xlsx"):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM final_recommendation_data", conn)
    conn.close()

    # 表示したい理想の列リスト
    target_cols = ['科目番号', '科目名', '実施学期', '曜時限', 'keywords']
    
    # 実際にデータの中に存在する列だけを抽出する
    available_cols = [c for c in target_cols if c in df.columns]
    
    # もし「曜時限」が見つからない場合、似た名前（スペース付きなど）がないか探す
    if '曜時限' not in available_cols:
        potential_match = [c for c in df.columns if '曜時限' in c]
        if potential_match:
            available_cols.append(potential_match[0])

    df_export = df[available_cols].copy()

    def format_keywords(val):
        if not val: return ""
        try:
            # リスト形式の文字列をカンマ区切りに変換
            if isinstance(val, str) and val.startswith('['):
                parsed = ast.literal_eval(val)
                return ", ".join(parsed) if isinstance(parsed, list) else val
            return val
        except:
            return val

    if 'keywords' in df_export.columns:
        df_export['キーワード(確認用)'] = df_export['keywords'].apply(format_keywords)

    df_export.to_excel(output_path, index=False)
    print(f"✅ Excelファイル '{output_path}' を保存しました。")
    print(f"出力された列: {available_cols}")

# 実行
export_keywords_to_excel('instance/project.db')

✅ Excelファイル '授業キーワード確認リスト.xlsx' を保存しました。
出力された列: ['科目番号', '科目名', '実施学期', 'keywords', '曜時限_x']


In [34]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('instance/project.db')

# 1. 解析結果（topic_distributions）を読み込む
df_topics = pd.read_sql("SELECT * FROM topic_distributions", conn)

# 2. 元の全データ（courses）から時間割情報などを読み込む
df_raw = pd.read_sql("SELECT 科目番号, 実施学期, 曜時限, 単位数 FROM courses", conn)

# 3. 科目番号をキーにして合体させる
# これにより、1つの表に解析結果と時間割が揃います
df_final = pd.merge(df_topics, df_raw, on='科目番号', how='left')

# 4. 決定版テーブルとして保存
df_final.to_sql('final_recommendation_data', conn, if_exists='replace', index=False)

conn.close()
print("✅ 統合テーブル 'final_recommendation_data' を作成しました。")

✅ 統合テーブル 'final_recommendation_data' を作成しました。


In [35]:
import sqlite3
import pandas as pd
import ast

def export_keywords_to_excel(db_path, output_path="授業キーワード確認リスト.xlsx"):
    conn = sqlite3.connect(db_path)
    # 作成した統合テーブルから読み込み
    df = pd.read_sql("SELECT 科目番号, 科目名, 実施学期, 曜時限, keywords FROM final_recommendation_data", conn)
    conn.close()

    def format_keywords(val):
        if not val: return ""
        try:
            # 文字列化されたリスト ['A', 'B'] を "A, B" に変換
            parsed = ast.literal_eval(val) if isinstance(val, str) and val.startswith('[') else val
            return ", ".join(parsed) if isinstance(parsed, list) else val
        except:
            return val

    df['キーワード(確認用)'] = df['keywords'].apply(format_keywords)
    df.to_excel(output_path, index=False)
    print(f"✅ Excelファイル '{output_path}' を保存しました。")

# 実行
export_keywords_to_excel('instance/project.db')

DatabaseError: Execution failed on sql 'SELECT 科目番号, 科目名, 実施学期, 曜時限, keywords FROM final_recommendation_data': no such column: 曜時限

In [32]:
import sqlite3
import pandas as pd
import ast

def export_keywords_to_excel(db_path, output_path="授業キーワード確認リスト.xlsx"):
    conn = sqlite3.connect(db_path)
    query = "SELECT 科目番号, 科目名, 実施学期, 曜時限, keywords FROM topic_distributions"
    
    try:
        df = pd.read_sql(query, conn)
    except pd.io.sql.DatabaseError:
        query = "SELECT 科目番号, 科目名, 実施学期, 曜時限, keywords FROM processed_courses"
        df = pd.read_sql(query, conn)
        
    conn.close()

    def format_keywords(val):
        if not val: return ""
        try:
            # 文字列化されたリスト ['A', 'B'] を "A, B" に変換
            parsed = ast.literal_eval(val)
            return ", ".join(parsed) if isinstance(parsed, list) else val
        except:
            return val

    df['キーワード(確認用)'] = df['keywords'].apply(format_keywords)
    df.to_excel(output_path, index=False)
    print(f"✅ Excelファイル '{output_path}' を保存しました。")


In [33]:
# 実行
export_keywords_to_excel('instance/project.db')

DatabaseError: Execution failed on sql 'SELECT 科目番号, 科目名, 実施学期, 曜時限, keywords FROM processed_courses': no such column: keywords

In [6]:
import sys
# 1. 現在の実行環境に対して gensim をインストール
!{sys.executable} -m pip install gensim

# 2. 読み込みテスト
try:
    from gensim.models import Word2Vec
    print("✅ gensim の読み込みに成功しました！")
except ImportError:
    print("❌ インストールに失敗しました。プロンプトでの pip install を確認してください。")

✅ gensim の読み込みに成功しました！


In [7]:
import sqlite3
import pandas as pd
from gensim.models import Word2Vec
import numpy as np

# 1. データの読み込み
conn = sqlite3.connect('instance/project.db')
df = pd.read_sql("SELECT * FROM final_keywords", conn)

sentences = [str(k).split(',') for k in df['keywords'] if k]

# 2. Word2Vecモデルの作成（学習）独自のキーワード群から、言葉の意味空間を構築する
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# 3. トピック（A～F）を代表する単語の定義（例）
topic_keywords = {
    'A': '経営', 'B': '都市', 'C': '経済',
    'D': '数学', 'E': '最適化', 'F': 'データサイエンス'
}

def calculate_topic_score(keywords, target_topic_word):
    """授業のキーワード群とトピック代表語の類似度平均を算出"""
    words = str(keywords).split(',')
    scores = []
    for word in words:
        if word in model.wv and target_topic_word in model.wv:
            # Word2Vecによる類似度算出
            scores.append(model.wv.similarity(word, target_topic_word))
    return np.mean(scores) if scores else 0

# 4. 各トピックへの適合度計算
for label, word in topic_keywords.items():
    print(f"トピック {label} ({word}) の適合度を計算中...")
    df[f'score_{label}'] = df['keywords'].apply(lambda k: calculate_topic_score(k, word))

# 5. トピック比率の正規化
score_cols = [f'score_{label}' for label in topic_keywords.keys()]
df['total_score'] = df[score_cols].sum(axis=1)

for label in topic_keywords.keys():
    # 各トピックの比率を算出
    df[f'theta_{label}'] = df[f'score_{label}'] / df['total_score']
    df[f'theta_{label}'] = df[f'theta_{label}'].fillna(0)

# 6. 結果を保存
df.to_sql('topic_distributions', conn, if_exists='replace', index=False)
conn.close()

print("トピック分類および比率（θ_dk）の算出が完了しました！")

トピック A (経営) の適合度を計算中...
トピック B (都市) の適合度を計算中...
トピック C (経済) の適合度を計算中...
トピック D (数学) の適合度を計算中...
トピック E (最適化) の適合度を計算中...
トピック F (データサイエンス) の適合度を計算中...
トピック分類および比率（θ_dk）の算出が完了しました！


In [9]:
import sys
# 1. 現在の実行環境に対して pulp をインストール
!{sys.executable} -m pip install pulp

# 2. 読み込みテスト
try:
    import pulp
    print("✅ pulp の読み込みに成功しました！最適化ロジックを実行できます。")
except ImportError:
    print("❌ インストールに失敗しました。")

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   -----------------

## 上記のトピックの割り当ては仮のものであり、正しい名称は実際に中身を見て確認する必要がある。20260109

In [17]:
import sqlite3
import pandas as pd

def get_diverse_recommendations(db_path, top_n=20):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM topic_distributions", conn)
    conn.close()

    # 1. 総合スコア（ユーザーの関心度との適合度）でソート
    df_sorted = df.sort_values(by='score_total', ascending=False)

    # 2. 多様性を確保するための「トピック別・上位推薦」各トピック（A~F）で最もスコアが高いものを抽出
    topic_cols = [c for c in df.columns if c.startswith('theta_')]
    diverse_list = []
    
    for t_col in topic_cols:
        # そのトピックの比率が高い順にトップ3件をピックアップ
        top_in_topic = df.sort_values(by=t_col, ascending=False).head(3)
        diverse_list.append(top_in_topic)
    
    # 全てのピックアップを統合して重複を除去
    df_diverse = pd.concat(diverse_list).drop_duplicates(subset=['科目番号'])

    return df_sorted.head(top_n), df_diverse


In [19]:
import sqlite3
import pandas as pd
import numpy as np

def get_diverse_recommendations(db_path, top_n=20):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("SELECT * FROM topic_distributions", conn)
    conn.close()

    # 1. ユーザーの関心度 (l_k) の設定（仮の自分で設定した値）
    user_interest = {
        'theta_A': 0.3, 'theta_B': 0.1, 'theta_C': 0.1,
        'theta_D': 0.1, 'theta_E': 0.1, 'theta_F': 0.3
    }

    # 2. 総合適合スコア (score_total) の計算
    df['score_total'] = 0
    for t_col, l_k in user_interest.items():
        if t_col in df.columns:
            df['score_total'] += df[t_col] * l_k

    # 3. 適合度順のランキング
    df_sorted = df.sort_values(by='score_total', ascending=False)

    # 4. 多様性を確保するためのトピック別ピックアップ※よくわかんない
    topic_cols = [c for c in df.columns if c.startswith('theta_')]
    diverse_list = []
    
    for t_col in topic_cols:
        # 各トピックにおいて、そのトピックらしさが最も強く、かつスコアも高いもの
        top_in_topic = df.sort_values(by=[t_col, 'score_total'], ascending=False).head(3)
        diverse_list.append(top_in_topic)
    
    df_diverse = pd.concat(diverse_list).drop_duplicates(subset=['科目番号'])

    return df_sorted.head(top_n), df_diverse



In [20]:
# 実行
rank_df, diverse_df = get_diverse_recommendations('instance/project.db')
print("--- あなたへの適合度トップ5 ---")
print(rank_df[['科目名', 'score_total']].head())

--- あなたへの適合度トップ5 ---
          科目名  score_total
109     設計演習I     3.364934
234     設計演習I     3.364934
158  ネットワーク科学     0.715432
32   ネットワーク科学     0.715432
86     土地利用計画     0.695893


In [42]:
import pandas as pd
import sqlite3
import json
import unicodedata
from sudachipy import dictionary, tokenizer
from keybert import KeyBERT
from gensim import corpora, models

# 1. 初期化
tokenizer_obj = dictionary.Dictionary().create()
kw_model = KeyBERT()
stopwords = {"する", "ある", "なる", "の", "が", "です", "ます", "授業", "科目", "本講義"}

# 2. 前処理関数（修正版）
def preprocess_text_with_sudachi(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    mode = tokenizer.Tokenizer.SplitMode.C
    # n.part_of_speech()[0] で品詞の第一段階（名詞など）を取得
    tokens = [n.normalized_form() for n in tokenizer_obj.tokenize(text, mode)
              if n.part_of_speech()[0] in ["名詞"] 
              and n.normalized_form() not in stopwords]
    
    normalized_tokens = [unicodedata.normalize("NFKC", token) for token in tokens]
    return " ".join(normalized_tokens)

# 3. キーワード抽出関数
def extract_keywords_list(description):
    preprocessed_text = preprocess_text_with_sudachi(description)
    if preprocessed_text:
        # KeyBERTでトピックの元となる重要語を抽出
        keywords = kw_model.extract_keywords(preprocessed_text, top_n=15, keyphrase_ngram_range=(1, 1))
        return [kw[0] for kw in keywords]
    return []

# --- 実行フェーズ ---

# CSV読み込み (カラム名は適宜変更してください)
df = pd.read_csv('syllabus.csv')

# 全授業からキーワードリストを作成
# これがLDAの「入力文書」になります
df['keywords'] = df['description'].apply(extract_keywords_list)

# 4. LDAモデルの作成
dictionary_lda = corpora.Dictionary(df['keywords'])
corpus = [dictionary_lda.doc2bow(text) for text in df['keywords']]

num_topics = 15
lda_model = models.LdaModel(corpus=corpus, id2word=dictionary_lda, num_topics=num_topics, random_state=42)

# 5. 各授業のトピック比率（θ_dk）を計算してDB用フォーマットへ
def get_topic_weights(bow):
    weights = [0.0] * num_topics
    for topic_id, prob in lda_model.get_document_topics(bow, minimum_probability=0):
        weights[topic_id] = float(prob)
    return json.dumps(weights)

df['topic_weights'] = [get_topic_weights(b) for b in corpus]

# --- SQLiteへの保存 ---
conn = sqlite3.connect('course_database.db')
# 必要最低限のカラムを保存
df[['title', 'description', 'topic_weights']].to_sql('Courses', conn, if_exists='replace', index=False)
conn.close()

print("フェーズ1完了：トピック比率の計算とDB保存が成功しました。")

ModuleNotFoundError: Package `sudachidict_core` does not exist. You may install it with a command `$ pip install sudachidict_core`

In [40]:
import ast
import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pulp
from gensim import corpora, models
from gensim.models import CoherenceModel
from sklearn.metrics.pairwise import cosine_similarity

In [41]:
# 前処理
self.df_combined = self.df_combined.rename(columns={'キーワード': '1倍キーワード'})  # 'キーワード' 列をリスト形式に変換
#self.df_combined['キーワード'] = self.df_combined[f'{ratio}倍キーワード'].apply(ast.literal_eval)  # 'キーワード' 列をリスト形式に変換
self.df_combined['キーワード'] = self.df_combined[f'拡張キーワード'].apply(ast.literal_eval)
self.df_0 = self.df_0.iloc[6:]  # df_0 の最初5行を削除（英語の授業を除外）

NameError: name 'self' is not defined

In [ ]:
# 前処理
        

        # '授業科目名' 列をリスト化
        grad_course_list = pd.concat([self.df_0['授業科目名'], self.df_2['授業科目名']]).unique().tolist()
        social_course_list = self.df_1['授業科目名'].unique().tolist()

        # 大学院専門科目・大学院専門基礎科目の授業を抽出
        self.df_grad_courses = self.df_combined[self.df_combined['授業科目名'].isin(grad_course_list)].copy()
        self.df_social_courses = self.df_combined[self.df_combined['授業科目名'].isin(social_course_list)].copy()

        self.df_grad.rename(columns={'科目名 ': '科目名'}, inplace=True)

        # 評価を数値にマッピング
        grade_mapping = {'A+': 5, 'A': 4, 'B': 3, 'C': 2, 'D': 0, 'P': 3, 'F': 0}
        self.df_grad['総合評価'] = self.df_grad['総合評価'].replace(grade_mapping).astype('int')
        #self.df_grad['総合評価'] = self.df_grad.apply(lambda x: x['総合評価'] * int(x['単位数']))

        # 成績データから必要な列を抽出し、先頭の不要な行を除外
        self.df_grad = self.df_grad[['科目名', '総合評価']].iloc[9:]

        # '授業科目名' 列をデータフレームに変換して LEFT JOIN で結合
        self.df_result = self.df_combined[['授業科目名']]
        self.df_result = self.df_result.merge(self.df_grad[['科目名', '総合評価']], how='left', left_on='授業科目名', right_on='科目名')
        self.df_result['総合評価'] = self.df_result['総合評価'].fillna(0).astype(int)  # NaN を 0 に置換し整数型に変換
        self.df_result.drop(columns='科目名', inplace=True)  # 不要な列を削除

        self.df_0['シラバス'] = self.df_0['科目番号'].apply(return_syllabus_link)
        self.df_1['シラバス'] = self.df_1['科目番号'].apply(return_syllabus_link)
        self.df_2['シラバス'] = self.df_2['科目番号'].apply(return_syllabus_link)

        self.df_0['科目区分'] = [0]*len(self.df_0)
        self.df_2['科目区分'] = [1]*len(self.df_2)
            
        grad_subject_num_dict = (
              self.df_0.set_index('授業科目名')['科目番号'].astype(str).to_dict() |
              self.df_2.set_index('授業科目名')['科目番号'].astype(str).to_dict()
          )
        social_subject_num_dict = self.df_1.set_index('授業科目名')['科目番号'].astype(str).to_dict()
        
        social_subject_overview = self.df_1.set_index('授業科目名')['授業概要'].astype(str).to_dict()

        grad_subject_schedule = (
              self.df_0.set_index('授業科目名')['時間割'].astype(str).to_dict() |
              self.df_2.set_index('授業科目名')['時間割'].astype(str).to_dict()
          )

        grad_subject_unit = (
              self.df_0.set_index('授業科目名')['単位数'].astype(str).to_dict() |
              self.df_2.set_index('授業科目名')['単位数'].astype(str).to_dict()
          )

        # '関連授業分類' カラムを追加して分類を適用
        self.df_grad_courses['科目番号'] = self.df_grad_courses['授業科目名'].map(grad_subject_num_dict)
        self.df_grad_courses['時間割'] = self.df_grad_courses['授業科目名'].map(grad_subject_schedule)
        
        self.df_grad_courses['単位数'] = self.df_grad_courses['授業科目名'].map(grad_subject_unit).apply(lambda x: int(x[0]))
        self.df_grad_courses['実施学期'] = self.df_grad_courses['時間割'].apply(lambda x: x.split(" ")[0])
        self.df_grad_courses['曜時限'] = self.df_grad_courses['時間割'].apply(lambda x: x.split(" ")[1])
        self.df_grad_courses['学位プログラム'] = self.df_grad_courses['科目番号'].apply(classify_graduate_course)
        self.df_grad_courses['科目区分'] = self.df_grad_courses['科目番号'].apply(classify_graduate_basis)
        self.df_grad_courses['科目区分名'] =self.df_grad_courses['科目区分'].apply(lambda x: '専門科目' if x==1 else '専門基礎科目')
        self.df_grad_courses['シラバス'] = self.df_grad_courses['科目番号'].apply(return_syllabus_link)

        self.df_social_courses['科目番号'] = self.df_social_courses['授業科目名'].map(social_subject_num_dict)
        self.df_social_courses['主専攻'] = self.df_social_courses['科目番号'].apply(classify_social_course)
        self.df_social_courses['授業概要'] = self.df_social_courses['授業科目名'].map(social_subject_overview)
        self.df_social_courses['シラバス'] = self.df_social_courses['科目番号'].apply(return_syllabus_link)
    
    def create_lda_model(self):

        # ===== LDA モデルの作成 =====
        texts = self.df_combined['キーワード']  # トークン化されたキーワードのリスト
        self.dictionary = corpora.Dictionary(texts)
        corpus = [self.dictionary.doc2bow(text) for text in texts]
        self.lda_model = models.LdaModel(corpus=corpus, id2word=self.dictionary, num_topics=self.num_topics, random_state=42, passes=10)

        # ユーザーの成績評価リストを取得
        user_ratings = self.df_result['総合評価'].tolist()
        

        # ===== ユーザープロファイル作成 =====
        topic_distributions = [self.lda_model.get_document_topics(doc, minimum_probability=0) for doc in corpus]
        print(user_ratings)
        print(topic_distributions)
        # topic_distributions の行数（ドキュメントの数）を確認
        num_documents = len(topic_distributions)
        print("Number of documents:", num_documents)

        # user_ratings の長さを確認
        num_ratings = len(user_ratings)
        print("Number of user ratings:", num_ratings)

        # 両者が一致するかどうかを確認
        if num_documents == num_ratings:
            print("The lengths match. You can proceed with the averaging.")
        else:
            print("The lengths do not match. You need to adjust the user_ratings.")

        self.user_profile = np.average(
            np.array([[prob for _, prob in doc] for doc in topic_distributions]),
            axis=0,
            weights=user_ratings
        )
        print("\nユーザーの関心トピック分布:", self.user_profile)
        self.user_profile_percent = (self.user_profile / self.user_profile.sum() * 100).astype(int)
        
    def get_keywords_list(self):
        # キーワード列のすべてのリストを結合
        all_keywords = []
        for keywords in self.df_combined['キーワード']:
            if isinstance(keywords, str):  # 文字列の場合のみ処理
                all_keywords.extend(eval(keywords))  # リストとして評価して追加
            elif isinstance(keywords, list):  # 既にリスト形式の場合
                all_keywords.extend(keywords)

        # キーワードの出現頻度をカウント
        keyword_counts = Counter(all_keywords)

        # 出現頻度順に並び替え
        sorted_keywords = keyword_counts.most_common()
        keywords_only = [keyword for keyword, count in sorted_keywords]
        return keywords_only

    def get_topic_keywords(self, keywords_only):
        # ===== トピックの重要キーワード =====
        # トピックごとの単語分布を取得
        def find_highest_topic_for_keyword(keyword, dictionary):
            highest_topic_id = None
            highest_probability = 0

            # 各トピックをループして確率を確認
            for topic_id, topic_terms in self.lda_model.show_topics(formatted=False, num_words=len(dictionary)):
                for term_id, probability in topic_terms:
                    if term_id == keyword and probability > highest_probability:
                        highest_topic_id = topic_id
                        highest_probability = probability

            return highest_topic_id, highest_probability

        topic_keywords 